In [ ]:
# 0. Install dependencies (run once):
#    pip install azure-search-documents azure-identity sentence-transformers pandas pyarrow torch

# 1. Imports and PyTorch compatibility patch
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType
)

def _patch_torch_get_default_device():
    """
    For dummies: If your torch version doesn't know about CUDA,
    this adds a helper so the model can still decide GPU vs CPU.
    """
    if not hasattr(torch, "get_default_device"):
        torch.get_default_device = lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu")

_patch_torch_get_default_device()

# 2. Configure Azure Cognitive Search connection
service_endpoint = "https://mi-search-service.search.windows.net"
admin_key        = "1XGx2SIrkWt99ISmvkHw3OgeOuVlp7tsf6EyOnaT51AzSeBcrFRZ"
index_name       = "wikipedia-chunks-embeddings-only"

# 3. Load a subset of the data for quick testing
df = (
    pd.read_parquet(
        r"C:\Users\ninic\…\wiki_hybrid_chunks_300.parquet",
        columns=["group_id", "chunk_id", "chunk_text"]
    )

)

# 4. Generate embeddings with SentenceTransformer
#    - Tokenizes text into sub-word tokens
#    - Applies Transformer layers (self-attention + feed-forward)
#    - Mean-pools token embeddings into a single fixed vector
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    df["chunk_text"].tolist(),
    convert_to_numpy=True,
    batch_size=64,         # For dummies: 128 texts -> 2 batches of 64 instead of 128 single calls
    show_progress_bar=True
)

# 5. Define index schema (no vector search config)
#    Fields:
#      * chunk_id: unique key
#      * chunk_text: searchable full-text
#      * group_id: filterable metadata
#      * vector: stored embedding for later retrieval
fields = [
    SimpleField(name="chunk_id", type=SearchFieldDataType.String, key=True),
    SearchableField(
        name="chunk_text",
        type=SearchFieldDataType.String,
        analyzer_name="en.lucene"  # Lucene analyzer for tokenization/stemming
    ),
    SimpleField(name="group_id", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        stored=True  # Store the embedding so it shows up in search results
    )
]

# 6. Create or reset the Azure Search index
idx_client = SearchIndexClient(service_endpoint, AzureKeyCredential(admin_key))
if index_name in [idx.name for idx in idx_client.list_indexes()]:
    idx_client.delete_index(index_name)  # Remove existing index for a clean start
index = SearchIndex(name=index_name, fields=fields)
idx_client.create_index(index)           # Actually create the index

# 7. Upload documents (embeddings + metadata) in batches
search_client = SearchClient(service_endpoint, index_name, AzureKeyCredential(admin_key))
batch, batch_size = [], 100
for i, row in df.iterrows():
    batch.append({
        "chunk_id":   row["chunk_id"],
        "chunk_text": row["chunk_text"],
        "group_id":   row["group_id"],
        "vector":     embeddings[i].tolist()
    })
    if len(batch) >= batch_size:
        search_client.upload_documents(documents=batch)
        batch.clear()

# Upload any remaining documents
if batch:
    search_client.upload_documents(documents=batch)

print(f"Index '{index_name}' is ready with {df.shape[0]} embeddings stored.")
# For dummies: 500 docs + batch_size=100 => 5 full uploads, no leftovers if exactly 500.
